In [47]:
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
import cvxpy as cp
import pickle
import time


import sys
from pathlib import Path
from pyprojroot import here

ROOT_DIR = here()
FORECASTING_DIR = ROOT_DIR / "4_forecasting"
DATA_DIR = ROOT_DIR / "1_data" / "processed"
COPULA_DIR = ROOT_DIR / "5_scenario_gen"
MODEL_DIR = ROOT_DIR / "6_models"

sys.path.insert(0, str(FORECASTING_DIR))  # make forecasting module importable
sys.path.insert(0, str(COPULA_DIR))  # make copula module importable
sys.path.insert(0, str(MODEL_DIR))  # make model modules importable

from dispatch_layer import default_fixed_params, build_problem, solve_plain, make_layer
from dispatch_wrapper import get_prices, realised_breakdown, cholesky_of_second_moment, realised_cost
from forecasting import (reindex_and_impute, build_features, make_windows,
                             normalise_hist, denormalise_y, Baseline_Forecaster)
from copula_lib import FrozenCopulaSampler

# ------------------------------------------------------------------ CONFIG
BOX_LEVELS = [(0.05, 0.95), (0.10, 0.90), (0.15, 0.85), (0.20, 0.80),
              (0.25, 0.75), (0.30, 0.70), (0.35, 0.65)]
CORNERS = [("single", 0.0), ("single", 1.0), ("dual", 0.0), ("dual", 1.0)]
N_SCEN = 64
DT = 1.0
MIN_BOX = 1e-4
SOLVER = cp.CLARABEL                     # plain solve; Gurobi also fine
SAT_TOL = 1e-6

# split boundaries (UTC); 2018 delivery blocks fully inside [TRAIN_START, VAL_START - 1h]
TRAIN_START = pd.Timestamp("2018-01-01 00:00:00+00:00")
VAL_START   = pd.Timestamp("2019-01-01 00:00:00+00:00")
ISSUE_HOUR  = 9
HORIZON     = 24
N_HIST      = 168                        # <-- set to your forecaster's lookback

HIST_COLS = ["prosumption", "solar_irrad", "panel_temp", "ambient_temp"]
FEAT_COLS = ["solar_irrad", "panel_temp", "ambient_temp"]
EXO_COLS  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]
PRICE_COLS = ["da", "imb", "up_reg_cost", "down_reg_cost"]

In [30]:
base = pd.read_csv(DATA_DIR / "df_full.csv", parse_dates=["datetime"]).set_index("datetime")
base = reindex_and_impute(base, HIST_COLS, freq="1h", warn_gap=6)
frame = build_features(base, feature_cols=FEAT_COLS)

windows = make_windows(
    frame, y_range=(TRAIN_START, VAL_START - pd.Timedelta(hours=1)),
    gate_aligned_only=True, issue_hour=ISSUE_HOUR,
    hist_cols=HIST_COLS, exo_cols=EXO_COLS, target_col="prosumption",
    price_cols=PRICE_COLS, n_hist=N_HIST, horizon=HORIZON,
)

# --- load FROZEN baseline forecaster + sampler (your loaders) ---

    # ---- reproducibility (record SEED in the checkpoint) ----
SEED = 20240801
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| seed:", SEED)


# Load forecaster
forecaster_checkpoint = torch.load(FORECASTING_DIR / "baseline_forecaster_best.pt", weights_only = False, map_location="cpu")
model_config = forecaster_checkpoint["model_config"]
model = Baseline_Forecaster(**model_config) 
model.load_state_dict(forecaster_checkpoint["state_dict"])
model.to(DEVICE)
model.eval()
sc = forecaster_checkpoint["scaler_stats"]
QUANTILE_LEVELS = forecaster_checkpoint["quantile_levels"]


# load sampler
bundle = pickle.load(open(COPULA_DIR / "frozen_copula.pkl", "rb"))
Z_corr = bundle["Z_corr"]
levels = bundle["quantile_levels"]
sampler = FrozenCopulaSampler(Z_corr, levels).to(DEVICE)
assert getattr(sampler, "S", N_SCEN) == N_SCEN, "sampler scenario count must equal N_SCEN"

[impute] 65 missing values across ['prosumption', 'solar_irrad', 'panel_temp', 'ambient_temp'] after reindex
[impute]   prosumption: 19 missing, longest run 11h   <-- LONG GAP (review)
[impute]   panel_temp: 46 missing, longest run 19h   <-- LONG GAP (review)
device: cpu | seed: 20240801


In [31]:
def make_forecast_fn(model, scaler_stats, device):
    model.eval()
    
    def forecast_fn(x_hist_day, x_fut_day):
        with torch.no_grad():
            xh = normalise_hist(np.asarray(x_hist_day), scaler_stats)      # (n_hist, C_hist)
            xh = torch.as_tensor(xh, dtype=torch.float32, device=device).unsqueeze(0)
            xf = torch.as_tensor(np.asarray(x_fut_day), dtype=torch.float32, device=device).unsqueeze(0)
            q_norm = model(xh, xf)                                          # (1, K, Q) normalised
            q_phys = denormalise_y(q_norm, scaler_stats)                    # affine -> physical MW
        return q_phys.squeeze(0).to(torch.float64)                         # (K, Q)
    return forecast_fn

In [32]:
def boxes_from_quantiles(quantiles, mean, quantile_levels, box_levels=BOX_LEVELS, min_box=MIN_BOX):
    """All box half-widths for one day, from the baseline quantiles + baseline mean anchor.
    Returns {box_level: (h_plus, h_minus)} as numpy. Matches dispatch_wrapper.compute_box
    but reuses the already-computed mean (one sampler pass per day)."""
    q = quantiles.detach().cpu().numpy() if torch.is_tensor(quantiles) else np.asarray(quantiles)
    m = mean.detach().cpu().numpy() if torch.is_tensor(mean) else np.asarray(mean)
    levels = np.asarray(quantile_levels, float)
    out = {}
    for (lo, hi) in box_levels:
        i_lo = int(np.argmin(np.abs(levels - lo)))
        i_hi = int(np.argmin(np.abs(levels - hi)))
        h_plus = np.clip(q[:, i_hi] - m, min_box, None)
        h_minus = np.clip(m - q[:, i_lo], min_box, None)
        out[(lo, hi)] = (h_plus, h_minus)
    return out

forecast_fn = make_forecast_fn(model, sc, DEVICE)

In [33]:
d = 0

mean, xi = sampler.mean_and_errors(forecast_fn(windows.x_hist[d], windows.x_fut[d]))      # a real 2018 day
M = xi.detach().cpu().numpy().T @ xi.detach().cpu().numpy() / 64
w = np.linalg.eigvalsh(0.5*(M+M.T))
print("min/max eig:", w.min(), w.max(), " cond:", w.max()/max(w.min(),1e-30))

min/max eig: 0.0007133449492533673 2.2673569162944656  cond: 3178.4859746573197


In [ ]:
fps = {k: default_fixed_params(k, num_scenarios=N_SCEN) for k in (0.0, 1.0)}
bundles = {(pm, k): build_problem(fps[k], pm) for pm, k in CORNERS}


day = windows.delivery_start[d]

realised = np.asarray(windows.y[d], float)                         # (T,) realised prosumption
price_day = np.asarray(windows.price[d], float)                    # (T, 4)

# ONE forecaster + sampler pass per day (baseline is frozen; box-level/corner-independent)
quantiles = forecast_fn(windows.x_hist[d], windows.x_fut[d])       # (K, Q) torch physical
mean, xi = sampler.mean_and_errors(quantiles)                      # mean (T,), xi (N,T)
mean_np = mean.detach().cpu().numpy()
xi_np = xi.detach().cpu().numpy()
Sigma = cholesky_of_second_moment(xi_np)                           # (T,T), for k=1
boxes = boxes_from_quantiles(quantiles, mean, QUANTILE_LEVELS)

prices = get_prices(price_day, "single")   



"""
for pm, k in CORNERS:
    print (f"Sweep for price: {pm}, k: {k}")
    fp = fps[k]
    bundle = bundles[(pm, k)]
    prices = get_prices(price_day, pm)                             # {pi_da, pi_imb} | {pi_da,lam_up,lam_dn}
    base_vals = {"pl_hat": mean_np, "Sigma_xi_chol": Sigma,
                    "xi_samples": xi_np, **prices}                    # superset; solve_plain selects
    for (lo, hi), (h_plus, h_minus) in boxes.items():
        vals = {**base_vals, "h_plus": h_plus, "h_minus": h_minus}
"""




'\nfor pm, k in CORNERS:\n    print (f"Sweep for price: {pm}, k: {k}")\n    fp = fps[k]\n    bundle = bundles[(pm, k)]\n    prices = get_prices(price_day, pm)                             # {pi_da, pi_imb} | {pi_da,lam_up,lam_dn}\n    base_vals = {"pl_hat": mean_np, "Sigma_xi_chol": Sigma,\n                    "xi_samples": xi_np, **prices}                    # superset; solve_plain selects\n    for (lo, hi), (h_plus, h_minus) in boxes.items():\n        vals = {**base_vals, "h_plus": h_plus, "h_minus": h_minus}\n'

In [35]:
boxes.items()
test_upper = np.asarray([0.16245083, 0.16154959, 0.16123265, 0.16139776, 0.17445216,
       0.21642657, 0.26752993, 0.32743721, 0.41613731, 0.57377429,
       0.79568377, 1.05867657, 1.18355341, 1.14191738, 0.94179258,
       0.73286847, 0.55319987, 0.41504065, 0.37568617, 0.31324367,
       0.27645378, 0.22965695, 0.180277  , 0.16783711], float)

test_lower = np.asarray([0.13922377, 0.12552259, 0.12171608, 0.13473123, 0.16119208,
       0.22576824, 0.39225773, 0.70771713, 1.0245463 , 1.21892314,
       1.39379248, 1.52492347, 1.61203556, 1.54312428, 1.41582277,
       1.26318546, 1.19838752, 1.25087866, 1.11036157, 0.87879868,
       0.64930935, 0.42356847, 0.27249371, 0.17363957])

In [36]:
test_upper - test_lower

array([ 0.02322706,  0.036027  ,  0.03951657,  0.02666653,  0.01326008,
       -0.00934167, -0.1247278 , -0.38027992, -0.60840899, -0.64514885,
       -0.59810871, -0.4662469 , -0.42848215, -0.4012069 , -0.47403019,
       -0.53031699, -0.64518765, -0.83583801, -0.7346754 , -0.56555501,
       -0.37285557, -0.19391152, -0.09221671, -0.00580246])

In [37]:
import time
# build one day's vals for single k=1, then:


base_vals = {"pl_hat": mean_np, "Sigma_xi_chol": Sigma,
                "xi_samples": xi_np, **prices}
vals = {**base_vals, "h_plus": test_upper, "h_minus": test_lower}


t=time.perf_counter()
out = solve_plain(bundles[("single",1.0)], vals, solver=cp.OSQP)
print("one k=1 solve:", time.perf_counter()-t, "status:", bundles[("single",1.0)].problem.status)

one k=1 solve: 0.3989946760048042 status: optimal


In [38]:
t=time.perf_counter()
out = solve_plain(bundles[("single",1.0)], vals, solver=cp.GUROBI)
print("one k=1 solve:", time.perf_counter()-t, "status:", bundles[("single",1.0)].problem.status)

one k=1 solve: 0.2929368400000385 status: optimal


In [39]:
loose = solve_plain(bundles[("single",1.0)], vals, solver=cp.OSQP)
tight = solve_plain(bundles[("single",1.0)], vals, solver=cp.GUROBI)   # the 43s one, but just once
for k in ["p_ch_hat","p_dis_hat","D_ch","D_dis","p_da_rel"]:
    print(k, np.abs(np.asarray(loose[k])-np.asarray(tight[k])).max())

p_ch_hat 0.00044859502575509715
p_dis_hat 0.0004125444883537351
D_ch 2.605651805431819e-09
D_dis 6.2816997992154455e-09
p_da_rel 0.00014763898767355954


In [40]:
for g in [1e-6, 1e-5, 1e-4, 1e-3]:
    fp = default_fixed_params(0, num_scenarios=64, gamma=g)
    b  = build_problem(fp, "single")
    t=time.perf_counter()
    loose = solve_plain(b, vals, solver=cp.OSQP, eps_abs=1e-6, eps_rel=1e-6, max_iter=50000, polish=True)
    tight = solve_plain(b, vals, solver=cp.CLARABEL)          # ground truth
    dt_ = time.perf_counter()-t
    gap = max(np.abs(np.asarray(loose[k])-np.asarray(tight[k])).max()
              for k in ["p_ch_hat","p_dis_hat","D_ch","D_dis","p_da_rel"])
    print(f"gamma={g:.0e}  clarabel+osqp time~{dt_:.2f}s  D/decision gap={gap:.2e}")

gamma=1e-06  clarabel+osqp time~0.24s  D/decision gap=7.76e-05
gamma=1e-05  clarabel+osqp time~0.24s  D/decision gap=7.76e-05
gamma=1e-04  clarabel+osqp time~0.27s  D/decision gap=7.76e-05
gamma=1e-03  clarabel+osqp time~0.24s  D/decision gap=7.77e-05


In [41]:
t=time.perf_counter(); solve_plain(b, vals, solver=cp.OSQP, eps_abs=1e-6, eps_rel=1e-6, max_iter=50000, polish=True); print("osqp", time.perf_counter()-t)
t=time.perf_counter(); solve_plain(b, vals, solver=cp.CLARABEL); print("clarabel", time.perf_counter()-t)

osqp 0.16062335300375707
clarabel 0.109497958997963


In [42]:
t=time.perf_counter(); solve_plain(b, vals, solver=cp.OSQP, eps_abs=1e-6, eps_rel=1e-6, max_iter=50000, polish=True); print("osqp", time.perf_counter()-t)
b.problem.solver_stats.num_iters

osqp 0.15465689800475957


1350

In [43]:
t=time.perf_counter(); solve_plain(b, vals, solver=cp.ECOS); print("ecos", time.perf_counter()-t)

ecos 0.10730962700472446


In [44]:
b.problem.solver_stats.num_iters

17

In [45]:
solvers = [(cp.CLARABEL, "CLARABEL"), (cp.ECOS,"ECOS") , (cp.SCS,"SCS")]


fp = default_fixed_params(1.0, num_scenarios=64, gamma=1e-4)
b  = build_problem(fp, "dual")            # dual k=1 = the heaviest k=1 (has epigraph? no—k=1 drops it; still, test dual)

for solverobject, solverstring in solvers:
    # plain solve
    t=time.perf_counter(); solve_plain(b, vals, solver=solverobject); print(f"For solver {solverstring}, k=1 plain: {time.perf_counter()-t}")
    # differentiable forward (what training runs)
    lay = make_layer(b)
    t=time.perf_counter()
    _ = lay(*[torch.tensor(np.asarray(vals[k],float)) for k in [p.name() for p in b.params]], solver_args={"solve_method":solverstring})
    print(f"For solver {solverstring}, k=1 layer forward: {time.perf_counter()-t}")

For solver CLARABEL, k=1 plain: 16.76550402700377
For solver CLARABEL, k=1 layer forward: 22.77688782000041
For solver ECOS, k=1 plain: 2.5143567600025563
For solver ECOS, k=1 layer forward: 1.350994503001857
For solver SCS, k=1 plain: 0.6917167999999947
For solver SCS, k=1 layer forward: 1.6745332149948808


In [ ]:
import time, torch
fp = default_fixed_params(1.0, num_scenarios=64, gamma=1e-4)
b  = build_problem(fp, "dual")
lay = make_layer(b)
keys = [p.name() for p in b.params]

# make the leaf we differentiate (proxy for the forecaster output) require grad
args = []
for k in keys:
    t = torch.tensor(np.asarray(vals[k], float), requires_grad=(k == "pl_hat"))
    args.append(t)

torch.cuda.synchronize() if torch.cuda.is_available() else None

for solverobject, solverstring in solvers[1:]:
    print(solverstring)
    t0 = time.perf_counter()
    dec = lay(*args, solver_args={"solve_method": solverstring})
    t1 = time.perf_counter()                       # forward done

    loss = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                     realised=realised, pl_hat=args[keys.index("pl_hat")],
                     price_model="dual", clip_recourse=True, **prices)
    loss.backward()
    t2 = time.perf_counter()                        # backward done

    print(f"forward:  {t1-t0:.3f}s")
    print(f"backward: {t2-t1:.3f}s")
    print(f"total:    {t2-t0:.3f}s")
    print(args[keys.index("pl_hat")].grad)


ECOS
forward:  0.579s
backward: 1.890s
total:    2.469s
tensor([43.5938, 55.0688, 33.5769, 17.9536, -6.9706, 27.8681, 37.5907, 31.0466,
        29.7991, 41.6881, 55.2430, 44.2647, 52.0060, 55.2602, 71.7461, 58.3074,
        58.3922, 72.8264, 57.7996, 44.6450, 54.6718, 50.0282, 39.2462, 35.0942],
       dtype=torch.float64)
SCS
forward:  1.117s
backward: 2.414s
total:    3.531s
tensor([ 87.1845, 110.1455,  67.1626,  35.9025, -13.9496,  55.7410,  75.1887,
         62.0885,  59.5921,  83.3756, 110.4914,  88.5228, 104.0077, 110.5209,
        143.4923, 116.6118, 116.7758, 145.6569, 115.5969,  89.2801, 109.3330,
        100.0391,  78.5633,  70.1637], dtype=torch.float64)


In [71]:
eps = 1e-4
g_fd = np.zeros(24)
base = realised_cost(fp, dec[0], dec[1], dec[2], dec[3], dec[4],
                     realised=realised, pl_hat=args[keys.index("pl_hat")],
                     price_model="dual", clip_recourse=True, **prices).item                     # scalar realised_cost at current pl_hat
for i in range(24):
    p = args[keys.index("pl_hat")].clone()
    p[i] += eps
    g_fd[i] = (args[keys.index("pl_hat")].grad[int(p)] - base).item / eps    # forward-difference; or central for accuracy
print(g_fd[:5])

ValueError: only one element tensors can be converted to Python scalars

In [ ]:
eps = 1e-4

# Extract target tensor and its key index once
pl_idx = keys.index("pl_hat")
pl_hat = args[pl_idx]

# 1. Base forward pass (added () to .item())
base = realised_cost(
    fp, dec[0], dec[1], dec[2], dec[3], dec[4],
    realised=realised, 
    pl_hat=pl_hat,
    price_model="dual", 
    clip_recourse=True, 
    **prices
).item()

# Initialize g_fd on the correct device/dtype
g_fd = torch.zeros(24, device=pl_hat.device, dtype=pl_hat.dtype)

for i in range(24):
    # Create perturbed vector
    p = pl_hat.clone()
    p[i] += eps
    
    # Substitute p into args for this iteration
    args_perturbed = list(args)
    args_perturbed[pl_idx] = p
    
    # 2. Run the layer with perturbed inputs
    # (Assuming decision outputs change with perturbed pl_hat)
    dec_perturbed = lay(*args_perturbed, solver_args={"solve_method": "ECOS"})
    
    # 3. Compute perturbed cost
    output = realised_cost(
        fp, dec_perturbed[0], dec_perturbed[1], dec_perturbed[2], dec_perturbed[3], dec_perturbed[4],
        realised=realised, 
        pl_hat=p,
        price_model="dual", 
        clip_recourse=True, 
        **prices
    ).item()
    
    # 4. Forward difference
    g_fd[i] = (output - base) / eps  

print(g_fd[:5])

tensor([-482.0075, -579.9391, -413.1616, -545.2283, -435.6777],
       dtype=torch.float64)


In [ ]:

tensor([-187.7882, -174.4818, -198.7844, -216.1193, -245.0888],
       dtype=torch.float64)

In [ ]:
eps = 1e-4
g_fd = np.zeros(24)
base = loss_value(pl_hat)                     # scalar realised_cost at current pl_hat
for i in range(24):
    p = pl_hat.clone(); p[i] += eps
    g_fd[i] = (loss_value(p) - base) / eps    # forward-difference; or central for accuracy
print(g_fd[:5])

tensor([43.5938, 55.0688, 33.5769, 17.9536, -6.9706, 27.8681, 37.5907, 31.0466,
        29.7991, 41.6881, 55.2430, 44.2647, 52.0060, 55.2602, 71.7461, 58.3074,
        58.3922, 72.8264, 57.7996, 44.6450, 54.6718, 50.0282, 39.2462, 35.0942],
       dtype=torch.float64)


In [57]:
args[keys.index("pl_hat")].grad.shape

torch.Size([24])

In [ ]:
prices = get_prices(price_day, "dual")   

